In [24]:
from pathlib import Path

# Source folder: ~/Documents/vid_dataset/
base_path = Path.home() / "Documents" / "vid_dataset" / "raw_vids"

# Destination folders
train_path = Path.home() / "Documents" /  "vid_dataset" / "split" / "train"
val_path = Path.home() / "Documents" /  "vid_dataset" / "split" / "val"
test_path = Path.home() / "Documents" /  "vid_dataset" / "split" / "test"

# Class names = folder names
classes = ["starlight", "wave", "spectrum_cycling"]

In [3]:
dataset_root_path = Path.home() / "Documents" /  "vid_dataset" / "split"

video_count_train = len(list(dataset_root_path.glob("train/*/*.mp4")))
video_count_val = len(list(dataset_root_path.glob("val/*/*.mp4")))
video_count_test = len(list(dataset_root_path.glob("test/*/*.mp4")))
video_total = video_count_train + video_count_val + video_count_test
print(f"Total videos: {video_total}")

Total videos: 232


In [4]:
dataset_root_path = Path.home() / "Documents" /  "vid_dataset" / "split"

all_video_file_paths = (
    list(dataset_root_path.glob("train/*/*.mp4"))
    + list(dataset_root_path.glob("val/*/*.mp4"))
    + list(dataset_root_path.glob("test/*/*.mp4"))
 )
all_video_file_paths[:5]


class_labels = sorted({path.parts[-2] for path in all_video_file_paths})
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Unique classes: {list(label2id.keys())}.")

Unique classes: ['spectrum_cycling', 'starlight', 'wave'].


In [5]:
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

model_ckpt = "MCG-NJU/videomae-base"
image_processor = VideoMAEImageProcessor.from_pretrained(model_ckpt)
model = VideoMAEForVideoClassification.from_pretrained(
    model_ckpt,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
)

C:\Users\Yin Yu\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
from torchvision.transforms import Compose, Resize, RandomCrop, RandomHorizontalFlip, Normalize

video_transform = Compose([
    Resize((224, 224)),
    RandomHorizontalFlip(),
    Normalize(mean=[0.45]*3, std=[0.225]*3),
])


In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_path,
    batch_size=4,
    num_workers=2,     
    pin_memory=True    
)

In [ ]:
from transformers import TrainingArguments, Trainer

model_name = model_ckpt.split("/")[-1]
new_model_name = f"{model_name}-finetuned-subset"
num_epochs = 3
batch_size = 4

args = TrainingArguments(
    new_model_name,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    num_train_epochs=num_epochs,
)

In [21]:
import os
import torch
import cv2
from pathlib import Path
from torch.utils.data import Dataset
from torchvision.transforms import functional as F

class VideoClassificationDataset(Dataset):
    def __init__(self, root_dir, class_names, num_frames=16, size=224, transform=None):
        self.root_dir = Path(root_dir)
        self.class_names = class_names
        self.class_to_idx = {cls: i for i, cls in enumerate(class_names)}
        self.num_frames = num_frames
        self.size = size
        self.transform = transform

        # Gather all (video_path, label_idx) pairs
        self.samples = []
        for class_name in class_names:
            class_folder = self.root_dir / class_name
            for file in class_folder.glob("*.mp4"):
                self.samples.append((file, self.class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = self.load_video_frames(video_path)
        frames = torch.stack([self.preprocess_frame(f) for f in frames])  # (T, C, H, W)

        return {
            "pixel_values": frames,  # required key for Hugging Face video models
            "labels": label,
        }

    def load_video_frames(self, path):
        cap = cv2.VideoCapture(str(path))
        frames = []
        while len(frames) < self.num_frames and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = torch.tensor(frame).permute(2, 0, 1) / 255.0  # (C, H, W), normalized
            frames.append(frame)
        cap.release()

        frames = frames[::3]  # 90 → 30

        # Pad or truncate
        if len(frames) < self.num_frames:
            pad = [torch.zeros_like(frames[0]) for _ in range(self.num_frames - len(frames))]
            frames.extend(pad)
        elif len(frames) > self.num_frames:
            frames = frames[:self.num_frames]
        return frames

    def preprocess_frame(self, frame):
        frame = F.resize(frame, [self.size, self.size])
        frame = F.normalize(frame, mean=[0.45] * 3, std=[0.225] * 3)
        return frame

class_names = ["wave", "starlight", "spectrum_cycling"]

train_dataset = VideoClassificationDataset(
    root_dir=train_path,
    class_names=class_names
)

val_dataset = VideoClassificationDataset(
    root_dir=val_path,
    class_names=class_names
)

In [22]:
import numpy as np
import evaluate
from transformers import Trainer

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset, 
    eval_dataset=val_dataset,    
    compute_metrics=compute_metrics,
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.211400,0.014694,1.000000
2,0.154500,0.000860,1.000000
3,0.000500,0.000428,1.000000
4,0.000300,0.000307,1.000000
5,0.000300,0.000246,1.000000
6,0.000200,0.000218,1.000000
7,0.000200,0.000201,1.000000
8,0.000200,0.000190,1.000000
9,0.000200,0.000184,1.000000
10,0.000200,0.000182,1.000000


TrainOutput(global_step=380, training_loss=0.039364101355711584, metrics={'train_runtime': 5152.2857, 'train_samples_per_second': 0.293, 'train_steps_per_second': 0.074, 'total_flos': 1.8815743100967322e+18, 'train_loss': 0.039364101355711584, 'epoch': 10.0})

In [25]:
trainer.evaluate()

{'eval_loss': 0.01469359640032053,
 'eval_accuracy': 1.0,
 'eval_runtime': 84.612,
 'eval_samples_per_second': 0.863,
 'eval_steps_per_second': 0.225,
 'epoch': 10.0}

In [26]:
test_dataset = VideoClassificationDataset(
    root_dir=test_path,
    class_names=class_names
)

# Predict and evaluate
predictions = trainer.predict(test_dataset)

# Show accuracy
print("Test Accuracy:", predictions.metrics["test_accuracy"])

# Optional: Show classification report
from sklearn.metrics import classification_report

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=class_names))


Test Accuracy: 1.0
                  precision    recall  f1-score   support

            wave       1.00      1.00      1.00         3
       starlight       1.00      1.00      1.00         3
spectrum_cycling       1.00      1.00      1.00         2

        accuracy                           1.00         8
       macro avg       1.00      1.00      1.00         8
    weighted avg       1.00      1.00      1.00         8



In [ ]:
base_path = Path.cwd().resolve().parents[1]
final_model_path = base_path / "models" / "video_recognition" / "transfer_model"

trainer.save_model(final_model_path)

No need to run once saved before

In [30]:
from transformers import VideoMAEImageProcessor

# Reuse the processor you loaded from the pretrained checkpoint
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")

# Save it alongside your model
processor.save_pretrained(final_model_path)

['C:\\Users\\Yin Yu\\Documents\\cv_automation\\models\\video_recognition\\transfer_model\\preprocessor_config.json']

Inference

In [38]:
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor

final_model_path = base_path / "models" / "video_recognition" / "transfer_model"
model = VideoMAEForVideoClassification.from_pretrained(final_model_path)
processor = VideoMAEImageProcessor.from_pretrained(final_model_path)

id2label = model.config.id2label  # e.g., {0: 'wave', 1: 'starlight', 2: 'spectrum_cycling'}
label2id = model.config.label2id  # inverse mapping

In [32]:
import cv2
import torch
import numpy as np

def extract_video_frames(video_path, num_frames=16, size=224):
    cap = cv2.VideoCapture(video_path)
    frames = []

    # Uniform sampling
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idxs = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    for idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        frames.append(frame)

    cap.release()
    frames = np.stack(frames)  # shape (T, H, W, C)
    return frames

In [42]:
from PIL import Image

def extract_video_frames(video_path, num_frames=16, size=224):
    cap = cv2.VideoCapture(video_path)
    frames = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idxs = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    for idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        pil_frame = Image.fromarray(frame)  # ✅ convert to PIL
        frames.append(pil_frame)

    cap.release()
    return frames  # ✅ List of 16 PIL Images


In [43]:
import torch.nn.functional as F

def predict_video(video_path):
    frames = extract_video_frames(video_path)
    inputs = processor(list(frames), return_tensors="pt")

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits  # shape: [1, num_classes]
        probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()  # shape: (num_classes,)

    # Print per-class probabilities
    print("Class Probabilities:")
    for idx, prob in enumerate(probs):
        print(f"{id2label[idx]}: {prob:.4f}")

    pred_id = np.argmax(probs)
    return id2label[pred_id]


In [44]:
print(model.config.label2id)
print(model.config.id2label)

{'spectrum_cycling': 0, 'starlight': 1, 'wave': 2}
{0: 'spectrum_cycling', 1: 'starlight', 2: 'wave'}


In [51]:
video_path = r"C:\Users\Yin Yu\Documents\cv_automation\Robot\pictures\testt_1.mp4"

print(predict_video(video_path))

Class Probabilities:
spectrum_cycling: 0.7484
starlight: 0.0709
wave: 0.1808
spectrum_cycling


In [ ]:
import matplotlib.pyplot as plt

frames = extract_video_frames(video_path)

for i, frame in enumerate(frames):
    plt.imshow(frame)
    plt.title(f"Frame {i}")
    plt.show()

Suggested code from hugging website

In [ ]:
from transformers import pipeline

video_cls = pipeline(model="")

In [ ]:
def predict_video(video_path, model, transform, num_frames=16):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while len(frames) < num_frames and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = torch.tensor(frame).permute(2, 0, 1) / 255.0
        frames.append(frame)
    cap.release()

    # Pad/truncate
    if len(frames) < num_frames:
        pad = [torch.zeros_like(frames[0]) for _ in range(num_frames - len(frames))]
        frames.extend(pad)
    else:
        frames = frames[:num_frames]

    frames = torch.stack([transform(f) for f in frames])  # (T, C, H, W)
    frames = frames.unsqueeze(0)  # Add batch dim

    with torch.no_grad():
        outputs = model(pixel_values=frames)
        pred = outputs.logits.argmax(dim=1).item()
    return id2label[pred]

# Example
result = predict_video("some_video.mp4", model, train_dataset.preprocess_frame)
print("Prediction:", result)
